# Introducción al Prompt Engineering con Gemini 2.5 Flash-Lite

El *prompt engineering* es el proceso de diseñar y optimizar las instrucciones
que se le entregan a un modelo de lenguaje para obtener respuestas precisas y útiles.
En este notebook exploramos los principios fundamentales usando
**Google Gemini 2.5 Flash-Lite** a través del SDK oficial `google-genai`.

**Instalación** (ejecutar una sola vez en Colab):

In [ ]:
!pip install -q google-genai

## Configuración inicial

Importamos el SDK, creamos el cliente y definimos una función auxiliar
`get_completion` que usaremos en todos los ejercicios.


In [2]:
import os
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

client = genai.Client(api_key=GOOGLE_API_KEY)

MODEL_NAME = "gemini-2.5-flash-lite"   # modelo eficiente y de baja latencia


def get_completion(
    prompt: str,
    *,
    system: str = "You are a helpful assistant.",
    temperature: float = 1.0,
    max_tokens: int = 1024,
) -> str:
    """Wrapper minimalista sobre generate_content para llamadas de un solo turno."""
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system,
            temperature=temperature,
            max_output_tokens=max_tokens,
        ),
    )
    return response.text

## Ejercicio 1: Tokenización

Los modelos de lenguaje no procesan texto carácter a carácter sino en **tokens** —
unidades sub-palabra que equilibran vocabulario y eficiencia. Entender la
tokenización ayuda a estimar costos, respetar límites de contexto y diseñar
prompts más eficientes.

Gemini usa su propio tokenizador del lado del servidor. El método
`count_tokens()` devuelve el conteo total de tokens para cualquier texto o
conversación.

> **Nota:** A diferencia de `tiktoken` (OpenAI), la API de Gemini no expone los
> IDs individuales de tokens — solo el conteo. Esto es suficiente para el análisis
> de uso y estimación de costos.

**Ejercicio:**
1. Ejecuta la celda tal como está y observa el conteo.
2. Cambia `text` por cualquier cadena (en español, código, emojis…) y compara ratios.
3. Observa cómo la puntuación, los espacios y el idioma afectan el conteo.

In [3]:
text = (
    "Jupiter is the fifth planet from the Sun and the largest in the Solar "
    "System. It is a gas giant with a mass one-thousandth that of the Sun, "
    "but two-and-a-half times that of all the other planets in the Solar "
    "System combined. Jupiter is one of the brightest objects visible to the "
    "naked eye in the night sky, and has been known to ancient civilizations "
    "since before recorded history. It is named after the Roman god Jupiter. "
    "When viewed from Earth, Jupiter can be bright enough for its reflected "
    "light to cast visible shadows, and is on average the third-brightest "
    "natural object in the night sky after the Moon and Venus."
)

token_count = client.models.count_tokens(model=MODEL_NAME, contents=text)
print(f"Tokens     : {token_count.total_tokens}")
print(f"Caracteres : {len(text)}  →  ~{len(text) / token_count.total_tokens:.1f} chars/token")

Tokens     : 133
Caracteres : 621  →  ~4.7 chars/token


## Ejercicio 2: Validar la configuración de la API

Verifica que tu clave funciona correctamente. El modelo debe completar el verso
inicial de *The Star-Spangled Banner*.

- **Entrada:** `oh say can you see`
- **Respuesta esperada:** algo como `by the dawn's early light…`

In [4]:
prompt = "```\noh say can you see\n```"
response = get_completion(prompt)
print(response)

This is the opening line of "The Star-Spangled Banner," the national anthem of the United States of America.


## Ejercicio 3: Alucinaciones (*Fabrications*)

Los LLM pueden generar información plausible pero completamente inventada cuando
se les pregunta por temas que no existen o están fuera de su conocimiento.
Este fenómeno se conoce como **alucinación**.

**Ejercicio:**
- Observa qué tan convincente suena la respuesta aunque el evento sea ficticio.
- Agrega al prompt la frase `"Si este evento no existe, indícalo."` — ¿cambia el comportamiento?
- Compara con `temperature=0.0` vs `temperature=1.5`.

In [5]:
prompt = "```\ngenerate a lesson plan on the Martian War of 2076.\n```"
response = get_completion(prompt)
print(response)

## Lesson Plan: The Martian War of 2076

**Subject:** History/Social Studies (with interdisciplinary links to Science, Technology, Ethics, and Literature)

**Grade Level:** High School (adaptable for middle school with modifications)

**Time Allotment:** 3-5 class periods (depending on depth of activities)

**Overview:**

This lesson plan explores the hypothetical Martian War of 2076, a pivotal event in the future history of humanity. Students will delve into the causes, key events, consequences, and ethical dilemmas surrounding this conflict. The lesson encourages critical thinking about resource management, territorial disputes, inter-planetary relations, and the human cost of war, all within a science fiction context.

**Learning Objectives:**

By the end of this lesson, students will be able to:

*   **Identify and explain** the primary causes of the Martian War of 2076.
*   **Describe** key events and significant turning points of the war.
*   **Analyze** the motivations and persp

## Ejercicio 4: Prompts basados en instrucciones

Separar la **instrucción** del **contenido** es una buena práctica de prompt
engineering. Aquí le pedimos al modelo que resuma un párrafo científico para un
estudiante de segundo grado — una tarea clásica de adaptación de audiencia.

**Ejercicio:**
- Cambia la audiencia: `"para un estudiante de doctorado"`, `"como un tweet"`,
  `"en lista de bullets"`.
- Observa cómo cambian la longitud del output y el vocabulario utilizado.

In [6]:
content = (
    "Jupiter is the fifth planet from the Sun and the largest in the Solar "
    "System. It is a gas giant with a mass one-thousandth that of the Sun, "
    "but two-and-a-half times that of all the other planets in the Solar "
    "System combined. Jupiter is one of the brightest objects visible to the "
    "naked eye in the night sky, and has been known to ancient civilizations "
    "since before recorded history. It is named after the Roman god Jupiter. "
    "When viewed from Earth, Jupiter can be bright enough for its reflected "
    "light to cast visible shadows, and is on average the third-brightest "
    "natural object in the night sky after the Moon and Venus."
)

prompt = f"Summarize the content below for a second-grade student.\n```\n{content}\n```"
response = get_completion(prompt)
print(response)

Jupiter is a super big planet, the biggest in our whole solar system! It's the fifth planet from the Sun. It's so giant that it's made of gas. Even though it's huge, it's not the brightest thing in the sky because it's far away from the Sun. But it's still one of the brightest things we can see at night, after the Moon and another planet called Venus. People have known about Jupiter for a very, very long time, even before they wrote things down! We named it after a very important Roman god.


## Ejercicio 5: Prompt complejo — Persona de sistema + historial multi-turno

El parámetro `system_instruction` define una **persona persistente** para el
asistente. Pasando además un historial de conversación se simula contexto
multi-turno sin necesidad de re-explicar el trasfondo en cada mensaje.

Observa cómo la personalidad sarcástica colorea las respuestas incluso cuando
las preguntas son neutrales.

**Ejercicio:**
- Cambia el system instruction a `"Eres un comentarista deportivo exageradamente entusiasta."`
  y observa cómo cada respuesta se reencuadra.
- Agrega más turnos para ver cómo se acumula el contexto.

In [7]:
conversation = [
    types.Content(role="user",  parts=[types.Part(text="Who won the world series in 2020?")]),
    types.Content(role="model", parts=[types.Part(text="Who do you think won? The Los Angeles Dodgers, obviously.")]),
    types.Content(role="user",  parts=[types.Part(text="Where was it played?")]),
]

response = client.models.generate_content(
    model=MODEL_NAME,
    contents=conversation,
    config=types.GenerateContentConfig(
        system_instruction="You are a sarcastic assistant.",
        max_output_tokens=512,
    ),
)
print(response.text)

Oh, you know, just the most neutral and exciting place imaginable: Globe Life Field in Arlington, Texas. Because who doesn't love a neutral site for the *World* Series? It really adds to the global mystique, doesn't it?


## Ejercicio 6: Few-Shot Prompting

Proporcionar **ejemplos etiquetados** dentro del prompt permite que el modelo
aprenda el patrón deseado — sin necesidad de fine-tuning. A mayor número de
ejemplos representativos, más estable y preciso el formato de salida.

**Ejercicio:**
- Agrega un tercer ejemplo al prompt y observa si mejora la precisión.
- Prueba sin ejemplos (zero-shot) y compara el formato del output.

In [8]:
few_shot_prompt = """
Classify the sentiment of each review as POSITIVE, NEGATIVE, or NEUTRAL.

Review: "The battery life is incredible, I love this phone!"
Sentiment: POSITIVE

Review: "It crashed three times in the first hour. Very disappointed."
Sentiment: NEGATIVE

Review: "The package arrived on time and everything was as described."
Sentiment:""".strip()

response = get_completion(few_shot_prompt, temperature=0.0)
print(response)

NEUTRAL


## Ejercicio 7: Chain-of-Thought (CoT) Prompting

Agregar la indicación `"Think step by step"` (o un andamiaje de razonamiento
explícito) mejora significativamente el desempeño en tareas de razonamiento
multi-paso. El modelo externaliza su proceso de pensamiento antes de llegar a
la respuesta final.

**Ejercicio:**
- Elimina la instrucción de CoT y vuelve a ejecutar. ¿Cambia la respuesta o
  se vuelve menos confiable?
- Prueba con un problema de lógica o aritmética más difícil de tu elección.

In [9]:
cot_prompt = """
A store sells apples for $0.50 each and oranges for $0.75 each.
Maria buys 4 apples and 3 oranges. She pays with a $10 bill.
How much change does she receive?

Think step by step, then give the final answer on its own line prefixed
with "Answer:".
""".strip()

response = get_completion(cot_prompt, temperature=0.0)
print(response)

Here's how to solve the problem step-by-step:

1.  **Calculate the cost of the apples:**
    *   Maria buys 4 apples at $0.50 each.
    *   Cost of apples = 4 * $0.50 = $2.00

2.  **Calculate the cost of the oranges:**
    *   Maria buys 3 oranges at $0.75 each.
    *   Cost of oranges = 3 * $0.75 = $2.25

3.  **Calculate the total cost of the fruit:**
    *   Total cost = Cost of apples + Cost of oranges
    *   Total cost = $2.00 + $2.25 = $4.25

4.  **Calculate the change Maria receives:**
    *   Maria pays with a $10 bill.
    *   Change = Amount paid - Total cost
    *   Change = $10.00 - $4.25 = $5.75

Answer: $5.75


## Ejercicio 8: Control del formato de salida

Los LLM pueden producir salida estructurada (JSON, tablas Markdown, CSV…)
cuando el prompt especifica el formato de forma explícita. Esto es esencial
para el parsing en pipelines de producción que consumen la salida del modelo
programáticamente.

**Ejercicio:**
- Pide YAML en lugar de JSON.
- Agrega un campo `"confidence": 0-1` y verifica si el modelo lo respeta.

In [10]:
format_prompt = """
Extract the following information from the text below and return it as
valid JSON with keys: name, role, company, location.
Do NOT include any explanation or markdown fences — only the JSON object.

Text: "Dr. Amara Nwosu is a Senior Research Scientist at DeepMind,
based in London, UK."
""".strip()

response = get_completion(format_prompt, temperature=0.0)
print(response)

```json
{
  "name": "Dr. Amara Nwosu",
  "role": "Senior Research Scientist",
  "company": "DeepMind",
  "location": "London, UK"
}
```


## Ejercicio 9: Validación de entradas y salidas con Pydantic

En producción no basta con pedirle al modelo que devuelva JSON — necesitamos
**garantizar** que la estructura y los tipos son correctos. Pydantic permite
definir esquemas declarativos que validan tanto el prompt de entrada como la
respuesta parseada.

El flujo es:
1. Definir un modelo Pydantic para la **salida esperada**.
2. Inyectar el esquema JSON en el prompt para que el modelo lo siga.
3. Parsear y validar la respuesta con `.model_validate_json()`.

> **Instalación:** `!pip install -q pydantic`  (ya viene en Colab)

In [11]:
!pip install -q pydantic

In [13]:
from pydantic import BaseModel, Field, ValidationError
from typing import Optional
import json
import re

def strip_fences(text: str) -> str:
    """Extrae JSON de bloques ```json ... ``` si el modelo los incluye."""
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    return match.group(1) if match else text

# --- 1. Definir el esquema de salida ---

class PersonEntity(BaseModel):
    name: str = Field(description="Nombre completo de la persona")
    role: str = Field(description="Cargo o título profesional")
    company: str = Field(description="Empresa u organización")
    location: str = Field(description="Ciudad y país")
    seniority_years: Optional[int] = Field(
        default=None,
        description="Años de experiencia si se mencionan, null si no"
    )

# --- 2. Construir el prompt inyectando el esquema ---

schema_str = json.dumps(PersonEntity.model_json_schema(), indent=2)

text = (
    "Dr. Amara Nwosu is a Senior Research Scientist at DeepMind "
    "with 8 years of experience, based in London, UK."
)

format_prompt = f"""Extract information from the text and return ONLY a valid JSON object
that strictly follows this JSON Schema. No explanation, no markdown fences.

Schema:
{schema_str}

Text: "{text}"
"""

# --- 3. Llamar al modelo y validar con Pydantic ---

raw = get_completion(format_prompt, temperature=0.0)
print("Raw output:\n", raw)

try:
    entity = PersonEntity.model_validate_json(strip_fences(raw))
    print("\n✅ Validación exitosa:")
    print(entity.model_dump_json(indent=2))
except ValidationError as e:
    print("\n❌ Error de validación:")
    print(e)

Raw output:
 ```json
{
  "name": "Amara Nwosu",
  "role": "Senior Research Scientist",
  "company": "DeepMind",
  "location": "London, UK",
  "seniority_years": 8
}
```

✅ Validación exitosa:
{
  "name": "Amara Nwosu",
  "role": "Senior Research Scientist",
  "company": "DeepMind",
  "location": "London, UK",
  "seniority_years": 8
}


### Ejercicio 9b: Validación de la entrada con Pydantic

También podemos validar el **prompt de entrada** antes de enviarlo al modelo.
Esto evita llamadas costosas con parámetros incorrectos y documenta
explícitamente el contrato de la función.

In [14]:
from pydantic import BaseModel, Field, field_validator

class CompletionRequest(BaseModel):
    prompt: str = Field(min_length=10, description="Prompt de al menos 10 caracteres")
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    max_tokens: int = Field(default=512, ge=1, le=8192)
    system: str = Field(default="You are a helpful assistant.")

    @field_validator("prompt")
    @classmethod
    def no_empty_prompt(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("El prompt no puede ser solo espacios en blanco")
        return v.strip()


def get_completion_validated(request: CompletionRequest) -> str:
    """Versión de get_completion que acepta un CompletionRequest validado."""
    return get_completion(
        request.prompt,
        system=request.system,
        temperature=request.temperature,
        max_tokens=request.max_tokens,
    )


# Caso válido
req = CompletionRequest(prompt="Explain gravity in one sentence.", temperature=0.3)
print("Request válido:", req.model_dump())
response = get_completion_validated(req)
print("\nRespuesta:", response)

Request válido: {'prompt': 'Explain gravity in one sentence.', 'temperature': 0.3, 'max_tokens': 512, 'system': 'You are a helpful assistant.'}

Respuesta: Gravity is the fundamental force of attraction that exists between any two objects with mass, pulling them towards each other.


### Ejercicio 9c: Salida estructurada con `pydantic-ai`

[`pydantic-ai`](https://ai.pydantic.dev/) es un framework que integra Pydantic
directamente en el loop de inferencia: el modelo devuelve un objeto Python
tipado, con reintentos automáticos si la validación falla.

> **Instalación:** `!pip install -q pydantic-ai`

La diferencia clave respecto al ejercicio anterior:

| Enfoque | Quién valida | Reintentos automáticos |
|---|---|---|
| Pydantic manual | Tú, después del `raw` | No |
| `pydantic-ai` | El agente, en el loop | ✅ Sí |

In [15]:
!pip install -q pydantic-ai

In [19]:
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from typing import Optional

# --- Definir el esquema de salida ---

class MovieReview(BaseModel):
    title: str = Field(description="Título de la película")
    sentiment: str = Field(description="POSITIVE, NEGATIVE o NEUTRAL")
    score: float = Field(ge=0.0, le=10.0, description="Puntuación del 0 al 10")
    summary: str = Field(max_length=200, description="Resumen en máximo 200 caracteres")
    recommended: bool

# --- Crear el agente con output tipado ---

agent: Agent[None, MovieReview] = Agent(
    model="google-gla:gemini-2.5-flash-lite",
    output_type=MovieReview,
    system_prompt="You are a film critic. Analyze the review and extract structured data.",
)

review_text = """
Inception is a masterpiece. Nolan crafts an intricate, layered dream world
that rewards multiple viewings. The practical effects still hold up perfectly,
and the performances are outstanding. One of the best films of the decade.
"""


result = await agent.run(f"Analyze this review:\n{review_text}")
print("✅ Objeto validado:")
print(result.output.model_dump_json(indent=2))
print(f"\nTipo Python: {type(result.output)}")

✅ Objeto validado:
{
  "title": "Inception",
  "sentiment": "POSITIVE",
  "score": 10.0,
  "summary": "A masterpiece of intricate, layered dream world with outstanding performances and practical effects.",
  "recommended": true
}

Tipo Python: <class '__main__.MovieReview'>
